## Tujuan Pembelajaran

* Membedakan local scope dan global scope, serta memahami bagaimana Python mencari nilai suatu variabel
* Menulis lambda (anonymous function) untuk kasus sederhana, dan tahu kapan sebaiknya tetap pakai def
* Mengenali mutable default argument bug — kenapa itu terjadi (menyambung ke materi 06), dan menerapkan pola aman untuk menghindarinya
* Mengenali kesalahan umum lain dalam menulis function, supaya tidak terjebak di kemudian hari

## Isi Materi

* Scope — local vs global, variable lookup
*  Lambda — anonymous function, kapan dipakai vs kapan pakai def
* Mutable Default Argument — bug klasik dan cara memperbaikinya
*  Common Function Mistakes — kumpulan kesalahan umum

## Goals

*  Menjelaskan kenapa variabel yang dibuat di dalam function tidak bisa diakses dari luar function
*  Menulis lambda sederhana dan menjelaskan kapan itu lebih cocok dibanding def
* Menjelaskan dengan kata sendiri kenapa def f(x, items=[]) itu berbahaya, memakai konsep dari materi 06
* Memperbaiki mutable default argument bug memakai pola None
* Mengenali function yang lupa return, terlalu bergantung pada global variable, atau terlalu kompleks

# 1. Scope

## Local scope dan local variabel

* Variabel yang dibuat di dalam function hanya "hidup" dan bisa diakses di dalam function itu saja
* Begitu function selesai dijalankan, variabel lokalnya "hilang" — tidak bisa diakses dari luar

In [5]:
def hitung_luas(panjang, lebar):
    luas = panjang * lebar   # 'luas' adalah local variable
    return luas

hasil = hitung_luas(5, 3)
print(hasil)   # 15

# print(luas)   -> ERROR! NameError: name 'luas' is not defined
# 'luas' cuma "hidup" selama function hitung_luas() sedang berjalan

15


## Global scope & global variable

* Variabel yang dibuat di luar function manapun (langsung di level utama script/notebook) disebut global variable
* Global variable bisa dibaca dari dalam function manapun tanpa perlu diteruskan sebagai parameter

In [6]:
nama_aplikasi = "Kalkulator Diskon"   # global variable

def tampilkan_info():
    print(f"Selamat datang di {nama_aplikasi}")   # bisa membaca variable global

tampilkan_info()   # Selamat datang di Kalkulator Diskon

Selamat datang di Kalkulator Diskon


## Variable lookup (secara sederhana)

* Saat Python menemukan sebuah nama variabel di dalam function, dia mencari nilainya dengan urutan: cek dulu di local scope (di dalam function itu sendiri) → kalau tidak ketemu, baru cek ke global scope

In [7]:
pesan = "Halo dari luar"   # global

def tampilkan():
    print(pesan)   # tidak ada 'pesan' lokal -> Python cari ke global, ketemu

tampilkan()   # Halo dari luar

Halo dari luar


Tapi hati-hati: membaca variabel global itu aman, sedangkan mengubah (assignment) variabel dengan nama yang sama di dalam function tidak mengubah yang global — malah membuat Python bingung

In [8]:
total_pesanan = 100000   # global

def tambah_pesanan():
    total_pesanan = total_pesanan + 50000   # ERROR!
    return total_pesanan

# tambah_pesanan()   -> UnboundLocalError: local variable 'total_pesanan'
#                        referenced before assignment
#
# Penyebabnya: begitu Python melihat ADA assignment ke 'total_pesanan'
# di dalam function ini, Python otomatis menganggap 'total_pesanan'
# sebagai variabel LOCAL di seluruh function — termasuk saat dibaca
# di sisi kanan tanda '+', padahal belum pernah didefinisikan secara lokal

Pola aman: daripada mengubah global secara langsung, kirim nilainya sebagai parameter dan kembalikan hasil barunya lewat return — ini persis prinsip input → process → output dari materi 09.

In [9]:
total_pesanan = 100000

def tambah_pesanan(total, tambahan):
    return total + tambahan

total_pesanan = tambah_pesanan(total_pesanan, 50000)
print(total_pesanan)   # 150000

150000


Kenapa pola ini lebih baik: function tambah_pesanan() jadi jelas apa yang dibutuhkan (parameter) dan apa yang dihasilkan (return value) — kamu tidak perlu membaca isi function untuk tahu efeknya. Ini yang disebut function predictable, dan akan dibahas lagi sebagai "kesalahan umum" di bagian 4.

# 2. Lambda

## Konsep anonymous function

* Lambda adalah cara membuat function tanpa nama, ditulis dalam satu baris, untuk logika yang sangat sederhana
* Fungsinya setara dengan def, hanya beda cara penulisan — dipakai saat kamu butuh function "sekali pakai" tanpa perlu repot memberi nama

## Syntax lambda

* Pola umum: lamda parameter: ekspresi
* Tidak ada return > hasil ekpresi otomatis jadi nilai kembalian


In [10]:
kuadrat_lambda = lambda x: x **2
print(kuadrat_lambda(5))


25


In [11]:
# Setara dengan def biasa:
def kuadrat(x):
    return x ** 2
print(kuadrat(5))

25


## Lambda Sederhana


lambda bisa menerima lebih dari satu parameter

In [12]:
tambah = lambda a, b: a+ b
print(tambah(3, 4))

7


Lambda bisa dikombinasikan dengan ternary (materi 07) untuk kondisi sederhana

In [14]:
cek_genap = lambda x: "Genap" if x % 2 == 0 else "Ganjil"
print(cek_genap(8))

Genap


## Kapan menggunakan lambda

* Paling umum dipakai sebagai argument sesaat untuk function lain yang butuh "function kecil" — contoh paling khas: parameter key pada sorted()

In [15]:
produk = [("Indomie", 3500), ("Beras", 12000), ("Telur", 28000)]

# urutkan berdasarkan harga (elemen index 1 tiap tuple, materi 04)
produk_urut = sorted(produk, key=lambda item: item[1])
print(produk_urut)
# [('Indomie', 3500), ('Beras', 12000), ('Telur', 28000)]

[('Indomie', 3500), ('Beras', 12000), ('Telur', 28000)]


 Cara Kerja `lambda item: item[1]` pada `sorted()`

Pada kode `sorted(produk, key=lambda item: item[1])`, `item` adalah **parameter** dari function `lambda` yang digunakan oleh `sorted()` untuk menentukan nilai dasar pengurutan. `item` tidak secara otomatis mengetahui variabel `produk`, tetapi `sorted()` yang mengambil setiap elemen dari `produk` lalu memasukkannya ke parameter `item` satu per satu. Karena setiap elemen `produk` berupa tuple `("Nama", harga)`, maka `item[1]` digunakan untuk mengambil **harga** sebagai nilai pembanding. Secara konsep, prosesnya adalah `("Indomie", 3500) → item[1] → 3500`, `("Beras", 12000) → item[1] → 12000`, dan `("Telur", 28000) → item[1] → 28000`, kemudian `sorted()` mengurutkan produk berdasarkan nilai tersebut.

```text
produk
  │
  ▼
sorted(produk, key=lambda item: item[1])
  │
  ├── ("Indomie", 3500) ──→ item ──→ item[1] ──→ 3500
  │
  ├── ("Beras", 12000) ───→ item ──→ item[1] ──→ 12000
  │
  └── ("Telur", 28000) ───→ item ──→ item[1] ──→ 28000
                                                   │
                                                   ▼
                                      Urutkan berdasarkan harga
```

## Kapan menggunakan def

* Begitu logikanya butuh lebih dari satu ekspresi (butuh beberapa baris, if/elif/else bertingkat, loop, dsb.), lambda tidak cocok — lambda secara sintaks hanya bisa menampung satu ekspresi tunggal
* Kalau function akan dipakai berkali-kali di banyak tempat, def dengan nama yang jelas jauh lebih mudah dibaca dibanding lambda tanpa nama yang tersebar di banyak tempa

In [16]:
# Ini TIDAK BISA ditulis sebagai lambda -> butuh banyak baris/kondisi bertingkat
def kategori_harga(harga):
    if harga < 5000:
        return "Murah"
    elif harga < 20000:
        return "Sedang"
    else:
        return "Mahal"

print(kategori_harga(3000))    # Murah
print(kategori_harga(15000))   # Sedang

Murah
Sedang


> Prinsip mengingat: lambda itu untuk kesederhanaan sesaat, bukan untuk menggantikan def di semua tempat. Kalau kamu merasa lambda-mu mulai susah dibaca (nested ternary di dalamnya, misalnya), itu sinyal untuk beralih ke def biasa — sama seperti prinsip menghindari nested ternary di materi 07.

# 3. Mutable Default Argument

## Masalahnya

In [17]:
def add_item(item, items=[]):   # BAHAYA! default value berupa list mutable
    items.append(item)
    return items

print(add_item("apel"))    # ['apel']                    -> sesuai ekspektasi
print(add_item("jeruk"))   # ['apel', 'jeruk']            -> LHO?! harusnya cuma ['jeruk']!
print(add_item("mangga"))  # ['apel', 'jeruk', 'mangga']  -> makin aneh!

['apel']
['apel', 'jeruk']
['apel', 'jeruk', 'mangga']


## Mengapa terjadi

* Default value pada parameter (items=[]) hanya dievaluasi satu kali, yaitu saat function pertama kali didefinisikan — bukan setiap kali function dipanggil
* Artinya: objek list [] itu dibuat sekali saja, lalu disimpan sebagai bagian dari definisi function
* Setiap kali kamu memanggil add_item() tanpa mengisi items, Python memakai objek list yang sama persis itu lagi — ini persis konsep shared reference dari materi 06!
* Karena list itu mutable, .append() mengubah objek itu secara permanen, dan perubahannya "menempel" untuk pemanggilan berikutnya

## Object default yang dipertahankan

buktikan dengan id(), teknik yang sama seperti materi 06:

In [18]:
def add_item(item, items=[]):
    print(f"id items: {id(items)}")   # buktikan objeknya SAMA tiap panggilan
    items.append(item)
    return items

add_item("apel")    # id items: 140712834958400 (misal)
add_item("jeruk")   # id items: 140712834958400 -> SAMA PERSIS!
add_item("mangga")  # id items: 140712834958400 -> tetap objek yang sama

id items: 2975280648832
id items: 2975280648832
id items: 2975280648832


['apel', 'jeruk', 'mangga']

> Koneksi ke materi 06: ingat kasus backup = data_asli yang "gagal jadi backup" karena shared reference? Kasus ini persis sama — bedanya, di sini yang "berbagi referensi" adalah objek default milik function itu sendiri, yang dipakai ulang diam-diam di setiap pemanggilan.

## Pola Aman

In [19]:
def add_item(item, items=None):
    if items is None:      # ingat: 'is None', bukan '== None' (aturan dari materi 06!)
        items = []           # list BARU dibuat setiap kali function dipanggil tanpa items
    items.append(item)
    return items

print(add_item("apel"))    # ['apel']
print(add_item("jeruk"))   # ['jeruk']    -> benar! tidak "menempel" dari panggilan sebelumnya
print(add_item("mangga"))  # ['mangga']

['apel']
['jeruk']
['mangga']


Pola ini juga berlaku sama untuk dict dan set sebagai default argument — keduanya sama-sama mutable

In [20]:
# Berbahaya juga:
def tambah_data(key, value, data={}):
    data[key] = value
    return data

# Pola aman yang sama:
def tambah_data_aman(key, value, data=None):
    if data is None:
        data = {}
    data[key] = value
    return data

> Prinsip mengingat: kalau kamu memberi default value pada parameter, dan default itu berupa list, dict, atau set — selalu curiga. Ganti jadi None dan buat objek barunya di dalam function.

# 4. Common Function Mistakes

## Lupa return

* Sudah disinggung di materi 09, ini kesalahan yang tetap sering terjadi bahkan setelah paham konsepnya — biasanya karena lupa, bukan karena tidak tahu

In [21]:
def hitung_total(a, b):
    total = a + b   # lupa return!

hasil = hitung_total(5, 3)
print(hasil)   # None -> bukan 8, karena function ini tidak mengirim nilai apa pun keluar

None


Salah membedakan parameter dan argument

* Ini lebih ke soal komunikasi teknis daripada bug — tapi penting supaya kamu bisa 
membaca dokumentasi/diskusi teknis dengan tepat
* Reminder dari materi 09: parameter = nama di definisi, argument = nilai saat dipanggil

In [22]:
def sapa(nama):     # 'nama' = PARAMETER
    print(f"Halo, {nama}")

sapa("Budi")          # "Budi" = ARGUMENT

Halo, Budi


## Global variable berlebihan

* Terlalu sering mengubah state lewat global variable membuat function sulit diprediksi — kamu harus baca seluruh isi function untuk tahu apa yang benar-benar diubahnya, bukan cukup baca parameter dan return-nya saja

In [23]:
# KURANG BAIK -> function ini punya "efek samping" tersembunyi
total_transaksi = 0

def tambah_transaksi(jumlah):
    global total_transaksi   # keyword untuk mengizinkan mengubah variabel global
    total_transaksi = total_transaksi + jumlah

tambah_transaksi(50000)
print(total_transaksi)   # 50000
# Masalahnya: untuk tahu function ini mengubah 'total_transaksi',
# kamu HARUS baca isi function-nya -- tidak cukup lihat cara memanggilnya saja

# LEBIH BAIK -> parameter + return (pola dari materi 09), lebih predictable
def tambah_transaksi_aman(total_saat_ini, jumlah):
    return total_saat_ini + jumlah

total_transaksi = 0
total_transaksi = tambah_transaksi_aman(total_transaksi, 50000)
print(total_transaksi)   # 50000

50000
50000


## Function terlalu kompleks

* Mengulang prinsip single responsibility dari materi 09 — function yang mengerjakan terlalu banyak hal sekaligus jadi sulit dites dan sulit dipakai ulang

In [24]:
# KURANG BAIK -> satu function melakukan validasi, hitung, DAN cetak sekaligus
def proses_pesanan(nama, harga, qty, diskon_persen):
    if harga < 0 or qty < 0:
        print("Data tidak valid")
        return None
    subtotal = harga * qty
    total = subtotal - (subtotal * diskon_persen / 100)
    print(f"Pesanan {nama}: Rp{total:,.0f}")
    return total

# LEBIH BAIK -> dipecah sesuai tanggung jawab masing-masing
def is_pesanan_valid(harga, qty):
    return harga >= 0 and qty >= 0

def hitung_total_pesanan(harga, qty, diskon_persen):
    subtotal = harga * qty
    return subtotal - (subtotal * diskon_persen / 100)

harga, qty, diskon = 25000, 3, 10
if is_pesanan_valid(harga, qty):
    total = hitung_total_pesanan(harga, qty, diskon)
    print(f"Total: Rp{total:,.0f}")

Total: Rp67,500


```text

Latihan

1. Lambda sederhana

Buat lambda untuk menghitung luas persegi panjang (panjang * lebar)
Buat lambda untuk mengecek apakah sebuah angka habis dibagi 3 (hasil True/False)
Diberikan data_siswa = [("Andi", 78), ("Budi", 92), ("Citra", 65)], gunakan sorted() dengan key=lambda ... untuk mengurutkan berdasarkan nilai (elemen kedua tuple), dari yang tertinggi

2. Identifikasi scope
Diberikan kode berikut:

python
batas_stok = 10

def cek_stok(stok):
    status = "Aman" if stok > batas_stok else "Menipis"
    return status

print(cek_stok(5))
print(status)
Sebelum menjalankan kode ini, tebak dulu: baris mana yang akan error, dan kenapa?
Jalankan kode-nya, cocokkan dengan tebakanmu
Jelaskan dengan kata sendiri kenapa batas_stok bisa dibaca dari dalam cek_stok(), tapi status tidak bisa dibaca dari luar

3. Memperbaiki mutable default argument
Diberikan function bermasalah berikut:

python
def tambah_peserta(nama, daftar_peserta=[]):
    daftar_peserta.append(nama)
    return daftar_peserta

kelas_a = tambah_peserta("Andi")
kelas_b = tambah_peserta("Budi")
Jalankan dulu kode ini, cetak kelas_a dan kelas_b — amati kenapa hasilnya tidak sesuai ekspektasi (harusnya kelas_a cuma berisi "Andi", kelas_b cuma berisi "Budi")
Perbaiki function tambah_peserta memakai pola None yang sudah dipelajari
Buktikan perbaikanmu benar dengan mencetak id() dari daftar_peserta di dua pemanggilan berbeda — pastikan berbeda id-nya (menandakan dua objek list yang independen)